# LoadCast: Exploratory Data Analysis (EDA)
## Historical Electricity Consumption Analysis for Rajasthan & Indian States

**Project:** LoadCast  
**Tagline:** Teaching a regression model to see tomorrow's electricity demand using yesterday's patterns.  
**Dataset:** Power Consumption in India (2019–2020) by twinkle0705  

### Objectives:
1. Inspect schema, data types, and temporal range.
2. Identify and resolve duplicate dates and continuity gaps.
3. Analyze Rajasthan electricity demand distribution, weekly periodicity, and seasonal patterns.
4. Assess structural break during the COVID-19 national lockdown (March 25 – May 31, 2020).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load raw dataset
df_raw = pd.read_csv('../data/raw/dataset_tk.csv')
df_raw.rename(columns={df_raw.columns[0]: 'Date'}, inplace=True)
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()[:10]}... total {len(df_raw.columns)}")

### Date Parsing & Deduplication
Dates are formatted as `DD/MM/YYYY HH:MM:SS`. 5 duplicate dates exist in July 2019 (July 8–12), which we resolve by computing daily mean consumption.

In [2]:
df_raw['ParsedDate'] = pd.to_datetime(df_raw['Date'], format='%d/%m/%Y %H:%M:%S')
dups = df_raw['ParsedDate'].duplicated().sum()
print(f"Duplicate date records: {dups}")

# Daily aggregation
numeric_cols = [c for c in df_raw.columns if c not in ['Date', 'ParsedDate']]
daily = df_raw.groupby('ParsedDate')[numeric_cols].mean().sort_index().reset_index()
daily.rename(columns={'ParsedDate': 'Date'}, inplace=True)
print(f"Deduplicated daily series shape: {daily.shape}")
print(f"Temporal coverage: {daily['Date'].min().strftime('%Y-%m-%d')} to {daily['Date'].max().strftime('%Y-%m-%d')}")

### Rajasthan Consumption Profile
Examine mean, median, peak, lowest demand, and standard deviation in Mega Units (MU).

In [3]:
raj = daily[['Date', 'Rajasthan']].copy()
raj.rename(columns={'Rajasthan': 'demand'}, inplace=True)
stats = raj['demand'].describe()
print("Rajasthan Demand Statistics (MU):")
print(stats)

### Time-Series Plot & COVID-19 Structural Break
Note the sharp contraction in electricity consumption starting around March 25, 2020 due to national lockdown measures.

In [4]:
plt.figure(figsize=(14, 5))
plt.plot(raj['Date'], raj['demand'], label='Daily Demand (MU)', color='#d97706', linewidth=1.5)
plt.axvspan(pd.to_datetime('2020-03-25'), pd.to_datetime('2020-05-31'), color='#ef4444', alpha=0.18, label='COVID-19 Lockdown')
plt.title('Rajasthan Electricity Consumption (2019-2020) [MU]')
plt.xlabel('Date')
plt.ylabel('Demand (MU)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()